Introduction here

## Initialization

In [ ]:
# Imports
from datetime import datetime
from typing import Literal, Callable

import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

from data_processing import loading, types, helpers
from data_processing import processing as proc
from data_processing import dataframe_validation as df_valid
from data_processing import experiment_data_keys as edk
from data_processing.processing import neutron_window_strategy as nws

### Functions

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[
    edk.ExperimentDataKey.CAEN_CALIBRATION,
    edk.ExperimentDataKey.NEW_CALIBRATION
]
NasaBorderKey = Literal[
    edk.ExperimentDataKey.NASA_BORDERS,
    edk.ExperimentDataKey.NASA_BORDERS_RECALC
]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        edk.ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else edk.ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{edk.ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                edk.ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else edk.ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = loading.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = loading.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: nws.NeutronStrategyFactory,
    window_type: types.WindowType,
    loading: bool,
    settings: types.NeutronWindowSettings
) -> Callable[[], nws.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: edk.ExperimentNeutronData, 
    factory_fn: Callable[[], nws.AbstractNeutronStrategy]
) -> edk.ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, edk.ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


## Analysis

### User Inputs

In [ ]:
experiment_ids = helpers.input_experiment_ids()

In [ ]:
calib_input = helpers.get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    df_valid.DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else df_valid.DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    edk.ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else edk.ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
detector_code = helpers.get_input_required(
    """\
Which detector was used?
1: Original detector (detector 1)
2: New detector (detector 2)
""",
    [proc.Detector.ZERO, proc.Detector.ONE],
    lambda x: proc.Detector(int(x)-1)
)

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
fit_input = helpers.get_input_with_default(
    """\
Which bimodal fit type do you want to use?
1: Bounds based
2: Peak finder based (default)
Press Enter for default
""",
    default_fit_input,
    int
)

fit_styles: dict[int, types.SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
done = False
strategy_factory = nws.NeutronStrategyFactory()

while not done:
    window_input = helpers.get_input_with_default(
        """\
Which neutron classification window do you want to use?
1: NASA window (default)
2: Neutron distribution window
Press Enter for default
""",
        1,
        int
    )
    load_window_input = helpers.get_input_with_default(
        """\
Do you want to load the borders from the standard border file?
[y/n, or press Enter for no]
""",
        "n",
        str
    )
    done = True
    will_load = load_window_input == "y"

    try:
        if window_input == 1:
            if will_load:
                settings = get_nasa_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", True, settings
                )
            else:
                settings = get_nasa_generation_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", False, settings
                )
                pass
        elif window_input == 2:
            if will_load:
                settings = get_n_distro_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", True, settings
                )
            else:
                settings = get_n_distro_generation_settings()
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", False, settings
                )
        else:
            print("Invalid classification window type given, please try again")
            done = False
    except ValueError as err:
        print("Problem found:")
        print(err)
        print("Please try again")
        done = False

experiment_neutron_data: edk.ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}
experiment_neutron_data = make_strategy_for_experiments(experiment_neutron_data, factory_fn)

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
}

### Something?

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[edk.ExperimentDataKey.UNCLASSIFIED] = loading.load_parquet_psd(exp_id)
    exp_data["signals_df"] = loading.load_parquet_signals(exp_id)

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[edk.ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, detector_code)
    exp_data[edk.ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3
overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[edk.ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[edk.ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[edk.ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[edk.ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[edk.ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[edk.ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[edk.ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[edk.ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[edk.ExperimentDataKey.END_SCAN_IDX]

    if fit_style == "bounds":
        # Default
        default_bounds: types.BimodalBounds = (
            types.BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
            types.BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
        )

        bounds_a: types.BimodalBounds = (
            types.BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            types.BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
        )

        bounds_b: types.BimodalBounds = (
            types.BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            types.BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
        )

        # Ranged Example
        bounds = [
            ((0, 60), bounds_a),
        ]
    else:
        default_bounds = None
        bounds = None

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style=fit_style,
        default_bounds=default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[edk.ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[edk.ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[edk.ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if edk.ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[edk.ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[edk.ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[edk.ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[edk.ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[edk.ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        df_valid.DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[edk.ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[edk.ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = df_valid.DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[edk.ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[edk.ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    neutrons_only = data_dict[edk.ExperimentDataKey.NEUTRONS_ONLY]
    gamma_only = data_dict[edk.ExperimentDataKey.GAMMA_ONLY]
    signals_df = data_dict["signals_df"]

    neutron_idx = neutrons_only.index[2]
    neutron_signal = signals_df.loc[neutron_idx].astype("int32")
    neutron_signal.index = neutron_signal.index.map(int)
    neutron_signal_baseline = neutron_signal.max()
    neutron_signal_peak = neutron_signal.min()
    neutron_peak_height = neutron_signal_baseline - neutron_signal_peak
    neutron_signal = -neutron_signal + neutron_signal_baseline

    # gamma_idx = gamma_only.index[0]
    # gamma_signal = signals_df.loc[gamma_idx].astype("int32")
    # gamma_signal.index = gamma_signal.index.map(int)
    gamma_signals = signals_df.loc[gamma_only.index].astype("int32")
    gamma_signals_baselines = gamma_signals.max(axis=1)
    gamma_signal_peaks = gamma_signals.min(axis=1)
    gamma_signal_height = gamma_signals_baselines - gamma_signal_peaks
    gamma_signal_matching = gamma_signals[gamma_signal_height.between(0.95 * neutron_peak_height, 1.05 * neutron_peak_height)]

    gamma_idx = gamma_signal_matching.index[0]
    gamma_signal = gamma_signals.loc[gamma_idx]
    gamma_signal.index = gamma_signal.index.map(int)
    gamma_signal_baseline = gamma_signals_baselines.loc[gamma_idx]
    gamma_signal = -gamma_signal + gamma_signal_baseline

    data_dict["neutron_signal"] = neutron_signal
    data_dict["gamma_signal"] = gamma_signal

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
window_size = 11
polyorder = 1
fontsize = 24
ticksize = 24

for exp_name, data_dict in experiment_neutron_data.items():
    neutron_signal = data_dict["neutron_signal"]
    gamma_signal = data_dict["gamma_signal"]

    neutron_x = neutron_signal.index
    neutron_y = savgol_filter(neutron_signal.to_numpy(), window_size, polyorder)
    # neutron_y = neutron_signal.to_numpy()
    gamma_x = gamma_signal.index
    gamma_y = savgol_filter(gamma_signal.to_numpy(), window_size, polyorder)
    # gamma_y = gamma_signal.to_numpy()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(neutron_x, neutron_y, "b-", label="Neutron")
    ax.plot(gamma_x, gamma_y, "r--", label="Gamma ray")
    # ax.legend()
    ax.set_xlabel("Analog-to-digital converter channel", fontsize=fontsize)
    ax.set_ylabel("Pulse height (arb. units)", fontsize=fontsize)
    ax.tick_params(axis="x", labelsize=ticksize)
    ax.tick_params(axis="y", labelsize=ticksize)
    fig.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()